# Step By Step 07 - Risk Factor Model
-------------------

> This example uses a data provider's factor model for alpha modeling.

## 1. Data Overview

In [ ]:
import datetime as dt


### 1.1 Factor Exposures

This section displays factor cross-sections for one day using only **style factors** and **industry factors**:

In [ ]:
import numpy as np


In [ ]:
import pyarrow as pa


In [ ]:
from alphamind.analysis.perfanalysis import perf_attribution_by_pos


### 1.2 Factor Risk Model

Similarly, I can retrieve the factor risk matrices:

In [ ]:
from alphamind.portfolio.meanvariancebuilder import mean_variance_builder


In [ ]:
from alphamind.portfolio.riskmodel import FactorRiskModel


#### 1.2.1 Factor Covariance

In [ ]:
codes = ["A", "B", "C"]


#### 1.2.2 Factor Loadings

These values are the same **factor exposures** shown in section 1.1.

In [ ]:
covariance = pa.table({
    "Factor": ["BETA", "SIZE"],
    "BETA": [0.02, 0.003],
    "SIZE": [0.003, 0.01],
})


#### 1.2.3 Idiosyncratic Factor Risk

In [ ]:
exposure = pa.table({
    "code": codes,
    "BETA": [1.0, 0.8, 1.2],
    "SIZE": [0.2, 0.4, -0.1],
})


## 2. Constructing an Alpha Signal with a Factor Model

First, construct a target factor using **calc-flow-python**:

In [ ]:
specific = pa.table({"code": codes, "idsync": [0.01, 0.015, 0.012]})


Factor exposures can remove **style effects** from the original factor through neutralization. This example uses only **market capitalization and Beta** style factors:

In [ ]:
model = FactorRiskModel(covariance, exposure, specific)
profile = model.get_risk_profile(codes)


I can verify that these two factors have been removed from the signal.

In [ ]:
benchmark = np.full(3, 1.0 / 3.0)


## 3. Portfolio Construction

This section uses the factor calculated above as a single-period forecast for portfolio construction.

The classic **single-period mean-variance model** is combined with common investment and trading constraints:

1. Leverage cannot be too low or too high.
2. Individual-stock concentration cannot be excessive.
3. Industry concentration cannot be excessive.
4. Style deviations must remain within specified preferences.
5. Maximize risk-adjusted returns subject to these constraints.

The following example covers all these points:

In [ ]:
status, objective, weights = mean_variance_builder(
    np.array([0.02, 0.03, 0.01]),
    profile,
    benchmark,
    0.0,
    1.0,
    np.ones((3, 1)),
    (np.array([1.0]), np.array([1.0])),
)


I apply the following controls:

1. Each stock's deviation from the benchmark cannot exceed 2%.
2. Every industry's deviation from the benchmark cannot exceed 3%.
3. Deviations for three style factors cannot exceed 50%.
4. Leverage must be between 80% and 120%.

The benchmark is normalized CSI 300 index weights. Users can instead use zero as the benchmark to create a long-short portfolio.

The most complex step is constructing the risk-control matrix and constraint bounds:

In [ ]:
# Individual-stock constraints
assert status in ("optimal", "optimal_inaccurate")


In [ ]:
# Use a constrained optimizer to solve this portfolio-optimization problem:
trade_date = dt.datetime(2026, 1, 5)


Verify each constraint individually:

1. Individual-Stock Limits

In [ ]:
active = pa.table({"trade_date": [trade_date] * 3, "code": codes, "weight": weights - benchmark})


2. Industry Deviations

In [ ]:
returns = pa.table({"trade_date": [trade_date] * 3, "code": codes, "return": [0.01, -0.02, 0.015]})


3. Style Deviations

In [ ]:
risks = pa.table({
    "trade_date": [trade_date] * 3,
    "code": codes,
    "BETA": exposure["BETA"],
    "SIZE": exposure["SIZE"],
})


4. Leverage

In [ ]:
attribution = perf_attribution_by_pos(active, returns, risks)


5. Utility

> **Note**: To save space, utility calculation is simplified here. Actual utility also includes a risk adjustment. Nevertheless, considering expected-return changes alone demonstrates the optimizer's effect.

In [ ]:
assert attribution.num_rows == 1


## 4. Epilogue

This example examined in detail how to use a risk-factor model for factor construction and portfolio optimization.

Alpha-mind can backtest these algorithms directly, so users do not need to implement every step manually.

The purpose of this example is to explain alpha-mind's factor-modeling principles and encourage customized development with its basic components. It cannot cover every factor-modeling and portfolio-optimization feature. Topics such as transaction-cost control are left for future chapters.

In [ ]:
attribution
